# 🤖 RAG Onboarding Chatbot
### Katering Yeyeti
**Final Project — AI Bootcamp**

Stack: LangChain · Groq (LLaMA 3.1) · Qdrant Cloud · SentenceTransformers · Streamlit

---

## Cell 1 — Install Libraries

In [40]:
!pip install langchain langchain-community langchain-text-splitters pymupdf sentence-transformers qdrant-client groq rouge-score -q

## Cell 2 — Mount Google Drive

In [41]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cell 3 — Konfigurasi Perusahaan
> ⚙️ **GANTI BAGIAN INI** sesuai perusahaan

In [42]:
# ============================================================
# ⚙️  KONFIGURASI — GANTI SESUAI PERUSAHAAN
# ============================================================
COMPANY_NAME    = "Katering Yeyeti"   # Nama perusahaan
FOLDER_PATH     = "/content/drive/MyDrive/Portfolio/Krama Translator/Katering Yeyeti" #"/content/drive/MyDrive/Colab Notebooks/FinalProject_AI_Bootcamp/Data/Input/Katering Yeyeti"
COLLECTION_NAME = "katering_yeyeti"   # Nama collection di Qdrant (huruf kecil, underscore)
# ============================================================

print(f"✅ Konfigurasi: {COMPANY_NAME}")
print(f"📁 Folder     : {FOLDER_PATH}")
print(f"🗄️  Collection : {COLLECTION_NAME}")

✅ Konfigurasi: Katering Yeyeti
📁 Folder     : /content/drive/MyDrive/Portfolio/Krama Translator/Katering Yeyeti
🗄️  Collection : katering_yeyeti


## Cell 4 — Load API Keys

In [43]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"]    = userdata.get("GROQ_API_KEY")
os.environ["QDRANT_URL"]      = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"]  = userdata.get("QDRANT_API_KEY")

print("✅ Semua API Key loaded!")

✅ Semua API Key loaded!


## Cell 5 — Load & Chunk PDF

In [44]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load PDF
pdf_files = [f for f in os.listdir(FOLDER_PATH) if f.endswith(".pdf")]
all_documents = []

for pdf_file in pdf_files:
    pdf_path = os.path.join(FOLDER_PATH, pdf_file)
    loader = PyMuPDFLoader(pdf_path)
    docs = loader.load()
    for doc in docs:
        doc.metadata["company"] = COMPANY_NAME
        doc.metadata["source_file"] = pdf_file  # tambahkan ini
    all_documents.extend(docs)
    print(f"✅ {pdf_file} ({len(docs)} halaman)")

print(f"\n📄 Total halaman: {len(all_documents)}")

# Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)
all_chunks = text_splitter.split_documents(all_documents)
print(f"🧩 Total chunks: {len(all_chunks)}")

✅ Incident_Report_YeyetiKatering.pdf (2 halaman)
✅ Data_Inventory_YeyetiKatering.pdf (2 halaman)
✅ Template_Absensi_YeyetiKatering.pdf (3 halaman)
✅ Sistem_Budaya_Kerja_YeyetiKatering.pdf (4 halaman)
✅ Profil_Perusahaan_YeyetiKatering.pdf (3 halaman)
✅ Kebijakan_Halal_YeyetiKatering.pdf (3 halaman)
✅ Surat_Edaran_YeyetiKatering.pdf (3 halaman)
✅ Memo_Internal_YeyetiKatering.pdf (3 halaman)
✅ Panduan_Customer_Complaint_YeyetiKatering.pdf (2 halaman)
✅ Form_Laporan_Harian_YeyetiKatering.pdf (2 halaman)
✅ Katalog_Menu_Produk_YeyetiKatering.pdf (4 halaman)

📄 Total halaman: 31
🧩 Total chunks: 145


## Cell 6 — Load Embedding Model

In [45]:
from sentence_transformers import SentenceTransformer

print("⏳ Loading embedding model...")
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding model loaded!")

⏳ Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Embedding model loaded!


## Cell 7 — Connect & Upload ke Qdrant Cloud

In [46]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, PayloadSchemaType

# Connect
qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
)
print("✅ Terhubung ke Qdrant Cloud!")

# Reset collection
if qdrant.collection_exists(COLLECTION_NAME):
    qdrant.delete_collection(COLLECTION_NAME)
    print("🗑️  Collection lama dihapus")

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

# Buat index untuk filter
qdrant.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="company",
    field_schema=PayloadSchemaType.KEYWORD
)
print("✅ Collection & index siap!")

# Embed & upload
texts     = [chunk.page_content for chunk in all_chunks]
metadatas = [chunk.metadata for chunk in all_chunks]
print(f"⏳ Embedding {len(texts)} chunks...")
embeddings = embedder.encode(texts, show_progress_bar=True)

points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={"text": texts[i], **metadatas[i]}
    )
    for i in range(len(texts))
]

qdrant.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"\n✅ {len(points)} chunks tersimpan di Qdrant Cloud!")

✅ Terhubung ke Qdrant Cloud!
🗑️  Collection lama dihapus
✅ Collection & index siap!
⏳ Embedding 145 chunks...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


✅ 145 chunks tersimpan di Qdrant Cloud!


## Cell 8 — Fungsi RAG Chat

In [47]:
from groq import Groq
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

# ============================================================
# 🧠 MULTI-TURN MEMORY — Riwayat percakapan disimpan di sini
# ============================================================
chat_history = []  # Format: [{"role": "user"|"assistant", "content": "..."}]
MAX_HISTORY_TURNS = 6  # Simpan maksimal 6 giliran terakhir (3 Q&A)

def reset_chat():
    """Reset riwayat percakapan ke awal."""
    global chat_history
    chat_history = []
    print("🔄 Riwayat percakapan direset.")

# ============================================================
# 📊 CONFIDENCE SCORE — Hitung keyakinan jawaban dari konteks
# ============================================================
def hitung_confidence(pertanyaan: str, results) -> dict:
    """
    Hitung confidence score berdasarkan:
    - avg_similarity : rata-rata skor similarity top chunks
    - top_similarity : skor chunk paling relevan
    - label          : LOW / MEDIUM / HIGH
    """
    if not results:
        return {"avg_similarity": 0.0, "top_similarity": 0.0, "label": "LOW"}

    # Embed ulang pertanyaan untuk dibandingkan dengan vektor chunks
    query_vec = embedder.encode(pertanyaan).reshape(1, -1)
    chunk_texts = [r.payload["text"] for r in results]
    chunk_vecs  = embedder.encode(chunk_texts)

    sims = cosine_similarity(query_vec, chunk_vecs)[0]
    avg_sim = float(np.mean(sims))
    top_sim = float(np.max(sims))

    if top_sim >= 0.75:
        label = "HIGH"
    elif top_sim >= 0.50:
        label = "MEDIUM"
    else:
        label = "LOW"

    return {"avg_similarity": round(avg_sim, 4), "top_similarity": round(top_sim, 4), "label": label}

# ============================================================
# 💬 FUNGSI UTAMA — rag_chat dengan Memory + Confidence Score
# ============================================================
def rag_chat(pertanyaan: str, top_k: int = 5, tampilkan_confidence: bool = True):
    global chat_history

    # 1. Embed pertanyaan
    query_vector = embedder.encode(pertanyaan).tolist()

    # 2. Cari chunks relevan di Qdrant
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
    ).points

    # 3. Hitung confidence score
    confidence = hitung_confidence(pertanyaan, results)

    # 4. Susun context dari chunks
    context_parts = []
    for i, r in enumerate(results):
        src = r.payload.get('source', 'Dokumen Internal')
        context_parts.append(f"[Chunk {i+1} | Sumber: {src}]\n{r.payload['text']}")
    context = "\n\n".join(context_parts)

    # 5. Beri peringatan otomatis di konteks jika confidence rendah
    confidence_hint = ""
    if confidence["label"] == "LOW":
        confidence_hint = "\n\n[SISTEM: Konteks yang ditemukan kurang relevan. Jawab hanya jika ada informasi yang cukup, atau akui keterbatasan.]"

    # 6. Susun messages: system + riwayat + pertanyaan baru
    system_prompt = f"""Kamu adalah Asisten Onboarding resmi di {COMPANY_NAME}, bertugas membantu karyawan baru memahami kebijakan dan informasi perusahaan.

## ATURAN UTAMA — WAJIB DIPATUHI
1. **HANYA gunakan informasi dari KONTEKS** yang diberikan. Dilarang keras menambahkan informasi dari pengetahuan umum atau asumsi pribadi.
2. **JANGAN mengarang atau berasumsi.** Jika informasi tidak ada di konteks, katakan dengan jelas: "Maaf, informasi tersebut tidak tersedia di dokumen internal yang saya miliki. Silakan hubungi HRD atau atasan langsung kamu untuk informasi lebih lanjut."
3. **Kutip sumber dokumen** (nama chunk/sumber) jika tersedia di konteks.
4. **Gunakan riwayat percakapan** untuk memahami konteks pertanyaan lanjutan (mis. "itu", "tadi", "sebelumnya").

## PANDUAN KUALITAS JAWABAN
- Gunakan bahasa Indonesia yang ramah, jelas, dan profesional.
- Jawab secara terstruktur: langsung ke inti, lalu detail pendukung.
- Jika ada daftar atau langkah-langkah, gunakan format bernomor atau berpoin.
- Jaga jawaban tetap ringkas namun lengkap.
- Akhiri dengan kalimat penutup yang membantu jika perlu.

## DETEKSI PERTANYAAN DI LUAR KONTEKS
- Jika pertanyaan tidak berkaitan dengan {COMPANY_NAME}, arahkan kembali dengan sopan.
- Jika konteks tidak cukup, akui keterbatasan secara jujur."""

    user_message = f"KONTEKS DOKUMEN:\n{context}{confidence_hint}\n\nPERTANYAAN:\n{pertanyaan}"

    messages = [
        {"role": "system", "content": system_prompt},
        *chat_history[-MAX_HISTORY_TURNS:],   # Sertakan riwayat (dibatasi)
        {"role": "user", "content": user_message}
    ]

    # 7. Generate jawaban
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages
    )
    jawaban = response.choices[0].message.content

    # 8. Simpan ke riwayat percakapan (tanpa context agar hemat token)
    chat_history.append({"role": "user",      "content": pertanyaan})
    chat_history.append({"role": "assistant", "content": jawaban})

    # 9. Tampilkan confidence score jika diminta
    if tampilkan_confidence:
        emoji = {"HIGH": "🟢", "MEDIUM": "🟡", "LOW": "🔴"}
        print(f"{emoji[confidence['label']]} Confidence: {confidence['label']} "
              f"(top={confidence['top_similarity']:.2f}, avg={confidence['avg_similarity']:.2f})")

    return jawaban, results, confidence

print(f"✅ Fungsi rag_chat ADVANCED siap untuk {COMPANY_NAME}!")
print("   Fitur aktif: 🧠 Multi-turn Memory | 📊 Confidence Score")

✅ Fungsi rag_chat ADVANCED siap untuk Katering Yeyeti!
   Fitur aktif: 🧠 Multi-turn Memory | 📊 Confidence Score


## Cell 9 — Test Chatbot

In [48]:
# ============================================================
# 🧪 TEST SINGLE PERTANYAAN
# ============================================================
reset_chat()  # Mulai sesi baru

pertanyaan = "Siapa kita dan apa yang kita lakukan?"
jawaban, sources, confidence = rag_chat(pertanyaan)

print(f"❓ Pertanyaan: {pertanyaan}")
print(f"\n💬 Jawaban:")
print("─" * 60)
print(jawaban)
print("─" * 60)
print(f"📚 Dari {len(sources)} chunks relevan")

🔄 Riwayat percakapan direset.
🔴 Confidence: LOW (top=0.38, avg=0.34)
❓ Pertanyaan: Siapa kita dan apa yang kita lakukan?

💬 Jawaban:
────────────────────────────────────────────────────────────
Saya bisa menjawab pertanyaan tersebut berdasarkan dokumen internal Karyawan Baru Yeyeti Katering.

Kita adalah karyawan Yeyeti Katering, sebuah perusahaan makanan yang memiliki filosofi "Dapur keluarga yang dibuka untuk umum". Kita bekerja seperti keluarga, dengan tujuan untuk membuat orang lain bahagia melalui makanan.

Kata Ibu Titi Karyani, pendiri Yeyeti, bahwa semua karyawan Yeyeti bukan sekadar pekerja, melainkan penjaga cita rasa dan penjaga kepercayaan pelanggan. Maka, kita harus bangga dengan apa yang kita lakukan dan pastikan bahwa makanan yang kita sajikan adalah layak untuk dijadikan pilihan bagi pelanggan.

Dalam prakteknya, kita akan bekerja dalam sistem kekeluargaan yang hangat dan saling percaya, dengan memahami bahwa kepercayaan pelanggan adalah tanggung jawab bersama kita. (Ch

In [49]:
# ============================================================
# TEST MULTI-TURN — Percakapan berlanjut (ingat konteks)
# ============================================================
# Jalankan cell ini BERKALI-KALI untuk simulasi percakapan.
# Chatbot akan ingat pertanyaan & jawaban sebelumnya.
# Jalankan reset_chat() untuk mulai sesi baru.

pertanyaan = input(f"❓ Tanya sesuatu tentang {COMPANY_NAME}: ")
jawaban, sources, confidence = rag_chat(pertanyaan)

print(f"\n💬 Jawaban:")
print("─" * 60)
print(jawaban)
print("─" * 60)
print(f"📚 Dari {len(sources)} chunks relevan")
print(f"🗂️  Riwayat tersimpan: {len(chat_history)//2} giliran")

❓ Tanya sesuatu tentang Katering Yeyeti: apa saja menu yang kita tawarkan?
🟡 Confidence: MEDIUM (top=0.64, avg=0.54)

💬 Jawaban:
────────────────────────────────────────────────────────────
Saya bisa menjawab pertanyaan tersebut berdasarkan dokumen internal Katalog Menu & Produk Yeyeti Katering.

Dalam katalog ini, disebutkan bahwa seluruh menu Yeyeti Katering dimasak segar setiap hari menggunakan bahan pilihan, bumbu rempah asli yang dihaluskan sendiri, tanpa MSG dan pengawet.

Beberapa menu yang ditawarkan adalah:

1. Nasi Box Reguler
2. Isian dari Nasi Box Reguler antara lain:
 * Nasi Cumi Hitam
 * Harga/Box: (harga tidak disebutkan tetapi disebutkan bahwa minimum order: 10 box untuk 1 menu)

Selain itu, disebutkan juga bahwa setiap paket katering yang dipesan akan mendapatkan Peyek Yeyeti sebagai pelengkap menu.
────────────────────────────────────────────────────────────
📚 Dari 5 chunks relevan
🗂️  Riwayat tersimpan: 2 giliran


In [50]:
# ============================================================
# 🔄 RESET SESI — Hapus riwayat percakapan
# ============================================================
reset_chat()

🔄 Riwayat percakapan direset.


## Cell 10 — EVALUASI

In [53]:
from rouge_score import rouge_scorer

# ============================================================
# 📊 DATA EVALUASI — Pertanyaan + Jawaban Referensi
# ============================================================
eval_data = [
    {
        "pertanyaan": "Apa visi dan misi perusahaan?",
        "referensi": "Visi dan misi perusahaan tercantum dalam dokumen profil perusahaan sebagai landasan operasional."
    },
    {
        "pertanyaan": "Apa saja benefit karyawan?",
        "referensi": "Karyawan mendapatkan benefit berupa kelonggaran kehadiran syar'i dan jaminan posisi selama cuti."
    },
    {
        "pertanyaan": "Bagaimana prosedur cuti karyawan?",
        "referensi": "Karyawan mengajukan cuti sesuai ketentuan yang berlaku dan posisi dijamin terjaga selama cuti."
    },
    {
        "pertanyaan": "Sebutkan produk atau layanan utama perusahaan.",
        "referensi": "Perusahaan menyediakan produk dan layanan utama sesuai bidang usaha yang tercantum di dokumen internal."
    },
    {
        "pertanyaan": "Apa saja peraturan kehadiran karyawan?",
        "referensi": "Karyawan wajib hadir sesuai jadwal kerja yang ditetapkan perusahaan dengan ketentuan absensi yang berlaku."
    },
]
# ============================================================

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

print(f"📊 Evaluasi ROUGE — {COMPANY_NAME}")
print("=" * 70)

total_r1, total_r2, total_rL = 0, 0, 0

for i, item in enumerate(eval_data):
    jawaban_chatbot, _, _ = rag_chat(item["pertanyaan"], tampilkan_confidence=False)
    scores = scorer.score(item["referensi"], jawaban_chatbot)

    r1 = scores['rouge1'].fmeasure
    r2 = scores['rouge2'].fmeasure
    rL = scores['rougeL'].fmeasure

    total_r1 += r1
    total_r2 += r2
    total_rL += rL

    print(f"\n❓ Q{i+1}: {item['pertanyaan']}")
    print(f"   ROUGE-1: {r1:.4f} | ROUGE-2: {r2:.4f} | ROUGE-L: {rL:.4f}")

n = len(eval_data)
print("\n" + "=" * 70)
print(f"📈 RATA-RATA ROUGE SCORE — {COMPANY_NAME}")
print(f"   ROUGE-1 : {total_r1/n:.4f}")
print(f"   ROUGE-2 : {total_r2/n:.4f}")
print(f"   ROUGE-L : {total_rL/n:.4f}")
print("=" * 70)

📊 Evaluasi ROUGE — Katering Yeyeti

❓ Q1: Apa visi dan misi perusahaan?
   ROUGE-1: 0.0966 | ROUGE-2: 0.0559 | ROUGE-L: 0.0690

❓ Q2: Apa saja benefit karyawan?
   ROUGE-1: 0.1143 | ROUGE-2: 0.0347 | ROUGE-L: 0.1029


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01ktfqx7nte3cskg9cf8sh1bwh` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 3306, Requested 2726. Please try again in 320ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}